# SpaceX Falcon 9 First Stage Landing Prediction
## Lab 4: Exploratory Data Analysis with SQL

The SpaceX launch records are loaded into a SQLite table named `SPACEXTBL` and explored with SQL queries.

In [1]:
import sqlite3
import pandas as pd

pd.set_option('display.max_colwidth', None)

In [2]:
df = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_2/data/Spacex.csv")

# Tidy up: drop empty rows, make sure the landing outcome column has no space in its name
df = df.dropna(how='all')
df.columns = [c.strip() for c in df.columns]
df = df.rename(columns={c: 'Landing_Outcome' for c in df.columns if c.replace(' ', '') == 'Landing_Outcome'})

# Store dates as YYYY-MM-DD text so they sort and compare correctly in SQL
first_date = str(df['Date'].iloc[0])
df['Date'] = pd.to_datetime(df['Date'], dayfirst=not first_date[:4].isdigit()).dt.strftime('%Y-%m-%d')
df['PAYLOAD_MASS__KG_'] = pd.to_numeric(df['PAYLOAD_MASS__KG_'], errors='coerce')

print(df.shape)
print(list(df.columns))
df.head()

(101, 10)
['Date', 'Time (UTC)', 'Booster_Version', 'Launch_Site', 'Payload', 'PAYLOAD_MASS__KG_', 'Orbit', 'Customer', 'Mission_Outcome', 'Landing_Outcome']


,Date,Time (UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome
0,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
1,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of Brouere cheese",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
3,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
4,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt


In [3]:
con = sqlite3.connect("my_data1.db")
df.to_sql("SPACEXTBL", con, if_exists='replace', index=False)

def run_sql(query):
    """Print a SQL query and return its result as a dataframe."""
    print(query.strip())
    return pd.read_sql_query(query, con)

### Task 1: Names of the unique launch sites

In [4]:
run_sql("""
SELECT DISTINCT Launch_Site
FROM SPACEXTBL;
""")

SELECT DISTINCT Launch_Site
FROM SPACEXTBL;


,Launch_Site
0,CCAFS LC-40
1,VAFB SLC-4E
2,KSC LC-39A
3,CCAFS SLC-40


### Task 2: Five records where the launch site begins with 'CCA'

In [5]:
run_sql("""
SELECT *
FROM SPACEXTBL
WHERE Launch_Site LIKE 'CCA%'
LIMIT 5;
""")

SELECT *
FROM SPACEXTBL
WHERE Launch_Site LIKE 'CCA%'
LIMIT 5;


,Date,Time (UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome
0,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
1,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of Brouere cheese",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
3,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
4,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt


### Task 3: Total payload mass carried by boosters launched by NASA (CRS)

In [6]:
run_sql("""
SELECT SUM(PAYLOAD_MASS__KG_) AS Total_Payload_Mass_kg
FROM SPACEXTBL
WHERE Customer = 'NASA (CRS)';
""")

SELECT SUM(PAYLOAD_MASS__KG_) AS Total_Payload_Mass_kg
FROM SPACEXTBL
WHERE Customer = 'NASA (CRS)';


,Total_Payload_Mass_kg
0,45596


### Task 4: Average payload mass carried by booster version F9 v1.1

In [7]:
run_sql("""
SELECT AVG(PAYLOAD_MASS__KG_) AS Average_Payload_Mass_kg
FROM SPACEXTBL
WHERE Booster_Version = 'F9 v1.1';
""")

SELECT AVG(PAYLOAD_MASS__KG_) AS Average_Payload_Mass_kg
FROM SPACEXTBL
WHERE Booster_Version = 'F9 v1.1';


,Average_Payload_Mass_kg
0,2928.4


### Task 5: Date of the first successful landing on a ground pad

In [8]:
run_sql("""
SELECT MIN(Date) AS First_Successful_Ground_Pad_Landing
FROM SPACEXTBL
WHERE Landing_Outcome = 'Success (ground pad)';
""")

SELECT MIN(Date) AS First_Successful_Ground_Pad_Landing
FROM SPACEXTBL
WHERE Landing_Outcome = 'Success (ground pad)';


,First_Successful_Ground_Pad_Landing
0,2015-12-22


### Task 6: Boosters with a successful drone ship landing and payload mass between 4000 and 6000 kg

In [9]:
run_sql("""
SELECT Booster_Version, PAYLOAD_MASS__KG_
FROM SPACEXTBL
WHERE Landing_Outcome = 'Success (drone ship)'
  AND PAYLOAD_MASS__KG_ > 4000
  AND PAYLOAD_MASS__KG_ < 6000;
""")

SELECT Booster_Version, PAYLOAD_MASS__KG_
FROM SPACEXTBL
WHERE Landing_Outcome = 'Success (drone ship)'
  AND PAYLOAD_MASS__KG_ > 4000
  AND PAYLOAD_MASS__KG_ < 6000;


,Booster_Version,PAYLOAD_MASS__KG_
0,F9 FT B1022,4696
1,F9 FT B1026,4600
2,F9 FT B1021.2,5300
3,F9 FT B1031.2,5200


### Task 7: Total number of successful and failed mission outcomes

In [10]:
run_sql("""
SELECT TRIM(Mission_Outcome) AS Mission_Outcome, COUNT(*) AS Total
FROM SPACEXTBL
GROUP BY TRIM(Mission_Outcome);
""")

SELECT TRIM(Mission_Outcome) AS Mission_Outcome, COUNT(*) AS Total
FROM SPACEXTBL
GROUP BY TRIM(Mission_Outcome);


,Mission_Outcome,Total
0,Failure (in flight),1
1,Success,99
2,Success (payload status unclear),1


In [11]:
run_sql("""
SELECT CASE WHEN Mission_Outcome LIKE 'Success%' THEN 'Success' ELSE 'Failure' END AS Outcome,
       COUNT(*) AS Total
FROM SPACEXTBL
GROUP BY Outcome;
""")

SELECT CASE WHEN Mission_Outcome LIKE 'Success%' THEN 'Success' ELSE 'Failure' END AS Outcome,
       COUNT(*) AS Total
FROM SPACEXTBL
GROUP BY Outcome;


,Outcome,Total
0,Failure,1
1,Success,100


### Task 8: Booster versions that carried the maximum payload mass

In [12]:
run_sql("""
SELECT Booster_Version, PAYLOAD_MASS__KG_
FROM SPACEXTBL
WHERE PAYLOAD_MASS__KG_ = (SELECT MAX(PAYLOAD_MASS__KG_) FROM SPACEXTBL);
""")

SELECT Booster_Version, PAYLOAD_MASS__KG_
FROM SPACEXTBL
WHERE PAYLOAD_MASS__KG_ = (SELECT MAX(PAYLOAD_MASS__KG_) FROM SPACEXTBL);


,Booster_Version,PAYLOAD_MASS__KG_
0,F9 B5 B1048.4,15600
1,F9 B5 B1049.4,15600
2,F9 B5 B1051.3,15600
3,F9 B5 B1056.4,15600
4,F9 B5 B1048.5,15600
5,F9 B5 B1051.4,15600
6,F9 B5 B1049.5,15600
7,F9 B5 B1060.2,15600
8,F9 B5 B1058.3,15600
9,F9 B5 B1051.6,15600


### Task 9: Failed drone ship landings in 2015, with month, booster version and launch site

In [13]:
run_sql("""
SELECT substr(Date, 6, 2) AS Month, Landing_Outcome, Booster_Version, Launch_Site
FROM SPACEXTBL
WHERE Landing_Outcome = 'Failure (drone ship)'
  AND substr(Date, 1, 4) = '2015';
""")

SELECT substr(Date, 6, 2) AS Month, Landing_Outcome, Booster_Version, Launch_Site
FROM SPACEXTBL
WHERE Landing_Outcome = 'Failure (drone ship)'
  AND substr(Date, 1, 4) = '2015';


,Month,Landing_Outcome,Booster_Version,Launch_Site
0,01,Failure (drone ship),F9 v1.1 B1012,CCAFS LC-40
1,04,Failure (drone ship),F9 v1.1 B1015,CCAFS LC-40


### Task 10: Landing outcomes between 2010-06-04 and 2017-03-20, ranked by count

In [14]:
run_sql("""
SELECT Landing_Outcome, COUNT(*) AS Outcome_Count
FROM SPACEXTBL
WHERE Date BETWEEN '2010-06-04' AND '2017-03-20'
GROUP BY Landing_Outcome
ORDER BY Outcome_Count DESC;
""")

SELECT Landing_Outcome, COUNT(*) AS Outcome_Count
FROM SPACEXTBL
WHERE Date BETWEEN '2010-06-04' AND '2017-03-20'
GROUP BY Landing_Outcome
ORDER BY Outcome_Count DESC;


,Landing_Outcome,Outcome_Count
0,No attempt,10
1,Success (drone ship),5
2,Failure (drone ship),5
3,Success (ground pad),3
4,Controlled (ocean),3
5,Uncontrolled (ocean),2
6,Failure (parachute),2
7,Precluded (drone ship),1


In [15]:
con.close()